In [1]:
# Install ChromaDB
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
import chromadb

# Create ChromaDB client
client = chromadb.Client()

# Create collection
collection = client.create_collection(name="travel_demo")

print("Collection created:", collection.name)
print("Current document count:", collection.count())

Collection created: travel_demo
Current document count: 0


In [3]:
collection.add(
    documents=[
        "Paris is famous for the Eiffel Tower and beautiful museums.",
        "The Maldives is known for clear blue water and tropical beaches.",
        "Tokyo combines modern technology with traditional Japanese culture.",
        "Switzerland is popular for its mountains, lakes, and scenic trains.",
        "Dubai is known for luxury shopping, skyscrapers, and desert adventures."
    ],

    metadatas=[
        {"category": "city"},
        {"category": "beach"},
        {"category": "city"},
        {"category": "nature"},
        {"category": "city"}
    ],

    ids=[
        "trip1",
        "trip2",
        "trip3",
        "trip4",
        "trip5"
    ]
)

print("Documents added!")
print("Total count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 86.2MiB/s]


Documents added!
Total count: 5


In [4]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(
    preview["ids"],
    preview["documents"]
):
    print(doc_id, "->", doc)

trip1 -> Paris is famous for the Eiffel Tower and beautiful museums.
trip2 -> The Maldives is known for clear blue water and tropical beaches.
trip3 -> Tokyo combines modern technology with traditional Japanese culture.


In [5]:
result = collection.get(
    ids=["trip1", "trip3"]
)

for doc_id, doc, meta in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")

trip1 | {'category': 'city'} | Paris is famous for the Eiffel Tower and beautiful museums.
trip3 | {'category': 'city'} | Tokyo combines modern technology with traditional Japanese culture.


In [6]:
results = collection.query(
    query_texts=[
        "places with famous landmarks and culture"
    ],
    n_results=2
)

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(
        f"{doc_id} | distance={distance:.4f} | {doc}"
    )

trip1 | distance=1.0685 | Paris is famous for the Eiffel Tower and beautiful museums.
trip4 | distance=1.1675 | Switzerland is popular for its mountains, lakes, and scenic trains.


In [7]:
results = collection.query(
    query_texts=[
        "a relaxing destination with water and beaches"
    ],
    n_results=3,
    where={"category": "beach"}
)

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(
        f"{doc_id} | distance={distance:.4f} | {doc}"
    )

trip2 | distance=0.9887 | The Maldives is known for clear blue water and tropical beaches.


In [8]:
collection.update(
    ids=["trip4"],

    documents=[
        "Switzerland is popular for its mountains, lakes, scenic trains, and hiking trails."
    ],

    metadatas=[
        {
            "category": "nature",
            "updated": True
        }
    ]
)

updated = collection.get(ids=["trip4"])

print(updated["documents"])
print(updated["metadatas"])

['Switzerland is popular for its mountains, lakes, scenic trains, and hiking trails.']
[{'updated': True, 'category': 'nature'}]


In [9]:
collection.upsert(
    documents=[
        "Tokyo combines modern technology, traditional temples, and Japanese culture.",
        "New Zealand is famous for dramatic landscapes and outdoor adventures."
    ],

    ids=[
        "trip3",
        "trip6"
    ]
)

print("Total count after upsert:", collection.count())

print(
    collection.get(
        ids=["trip3", "trip6"]
    )["documents"]
)

Total count after upsert: 6
['Tokyo combines modern technology, traditional temples, and Japanese culture.', 'New Zealand is famous for dramatic landscapes and outdoor adventures.']


In [10]:
collection.delete(
    ids=["trip2"]
)

print(
    "Total count after delete:",
    collection.count()
)

print(
    "Remaining ids:",
    collection.get()["ids"]
)

Total count after delete: 5
Remaining ids: ['trip1', 'trip3', 'trip4', 'trip5', 'trip6']


In [11]:
collection.delete(
    where={"category": "city"}
)

print(
    "Total count after category delete:",
    collection.count()
)

print(
    "Remaining ids:",
    collection.get()["ids"]
)

Total count after category delete: 2
Remaining ids: ['trip4', 'trip6']


In [12]:
final_state = collection.get()

print(
    "Final document count:",
    collection.count()
)

for doc_id, doc, meta in zip(
    final_state["ids"],
    final_state["documents"],
    final_state["metadatas"]
):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 2
trip4 | {'category': 'nature', 'updated': True} | Switzerland is popular for its mountains, lakes, scenic trains, and hiking trails.
trip6 | None | New Zealand is famous for dramatic landscapes and outdoor adventures.
